# L2 step 4d · The 3B's counts with no options shown (GPU T4)

**What:** the field 3B (c) answers the benchmark's fair building-counting questions **with no answer options**, and is asked for a number. Fair: the largest option is 20 or less (2,002 questions), where the middle option isn't always the right one (L2 step 4b).

**Why:** with options, the 3B is right on 57.45% of these questions, ahead of C4's map (42.2%) and of a rule that never sees the photo (40.9%) beyond chance (E7 step 3c, 25 Nov). This run shows whether it counts or only picks well among the options it is shown.

**How:** the same model, fix and image size as L2 step 4 (cells 1, 2 and 2b are that notebook's, unchanged but for the output folder, the smoke-test size and the runner-up switched off). Each question keeps the benchmark's own wording; the options are left out and the instruction asks for only a number. Greedy, at most 32 new tokens; the first number in the answer counts. A question with no number in its answer is scored wrong. The comparison with C4's counts is made on the PC from saved files (`l2-step4d-compare.ipynb`).

**The rule, fixed 25 Nov before the run.** On the fair questions, the 3B counts if both hold:
1. its number, snapped to the nearest option, is right more often than the blind rule (40.9%), beyond chance (exact McNemar, p < 0.05);
2. its number is closer to the true count than C4's raw count more often than not (sign test over the questions where the two differ, p < 0.05).

If 1 fails, its lead with options came from reading the options. If 1 holds and 2 fails, it counts no better than the map.

**The expectation, fixed 25 Nov before the run:**
- at least 95% of the answers contain a number;
- the exact count on 30–50% of questions, below its 57.45% with options, since options narrow the choice;
- snapped to the nearest option, right on 45–57%, so test 1 passes;
- a mean error of 2–3 buildings, below C4's raw count's 3.78, so test 2 passes.

**Run:** GPU T4 x2, internet on (the base model and the adapter come from Hugging Face). Inputs: `redwannewaz/disasterm3-bench-mirror-v2`, the account's private `l2-hf-write-token` dataset, and the output of `l2-step3-heldout-select` (the winner's name). About an hour.

In [1]:
# ── Cell 1: Environment: Kaggle's torch and transformers, current peft (as L2 step 2), and no torchao ──
# L0's merge (26 Oct) stopped on "Found an incompatible version of torchao": peft checks torchao when it puts an adapter
# on an fp16 model, as this notebook does. Nothing here uses torchao.
import importlib.metadata as _md
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft"], check=True)

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

assert importlib.util.find_spec("torchao") is None, "torchao is still installed: restart the session and run again"
assert int(_version("transformers").split(".")[0]) >= 5, "written for transformers 5 (Kaggle's 5.16.1 on 26 Oct)"
import torch
VERSIONS = {p: _version(p) for p in ("torch", "transformers", "peft")}
print(f"✓ {VERSIONS}, python {sys.version.split()[0]}; torchao: not installed")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 1, "Pick GPU T4 x2 (one is used)"


✓ {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'peft': '0.22.0'}, python 3.13.16; torchao: not installed
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration: the winner, the benchmark ─────────────────────────
import ast, glob, json, math, os, re, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import AutoProcessor

BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_REPO = "AbrarAlam/disasterm3-qwen25vl3b-l2"
BATCH, MAX_NEW_TOKENS = 8, 32
SMOKE_TEST = False             # True = 16 questions
TIME_LIMIT_HOURS = 11.0
OUT_DIR = Path("/kaggle/working/L2S4D")
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANSWERS_PATH = OUT_DIR / "answers.jsonl"
CUDA = torch.cuda.is_available()
EXPECTED_QUESTIONS = {"DTR": 353, "BBR": 2006, "BDC": 3583, "DRE": 1868, "DSR": 2007, "ORR": 858}
REF_NEW_7B_FULL = {"DTR": 76.77, "BBR": 19.99, "BDC": 30.31, "DRE": 33.03, "DSR": 40.46, "ORR": 34.62}   # L0 step 2
REF_BASE_7B = {"DTR": 66.6, "BBR": 4.7, "BDC": 34.2, "DRE": 29.3, "DSR": 28.3, "ORR": 23.9}              # DisasterM3 Table 2

import glob as _g, os as _o
for _f in _g.glob("/kaggle/input/**/hf_write_token.txt", recursive=True):   # runs started from the PC have no secrets
    _o.environ["HF_TOKEN"] = open(_f, encoding="utf-8").read().strip()

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

# ── The rules all L2 step 2 notebooks share (the same text in each) ─────────
import hashlib

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
MAX_PIXELS = 768 * 28 * 28       # the 3B's image cap: a 1024 px image reaches it at 756 px (729 tokens); also the soft labels'
HELD_OUT_EVERY = 20              # 1 after-photo in 20 is held out to choose between the two 3Bs (fixed 8 Nov)
ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT = 55764, 38271, 1962   # counted on the PC from train_release.json, 8 Nov

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def scene(e):
    """The after-photo a question is about. A relational question's boxed picture, '<after-photo>_<k>.png',
    counts as its after-photo, so the same scene never sits on both sides of the split."""
    if e.get("post_image_path"):
        return name_of(e["post_image_path"])
    return re.sub(r"_\d+(\.png)$", r"\1", name_of(e.get("image_path", "")))

def held_out(e):
    return int(hashlib.md5(scene(e).encode()).hexdigest(), 16) % HELD_OUT_EVERY == 0

def resolve_image_path(rel_path):
    """L2 step 1's rule: a manifest path -> its file in the mirror (None if missing)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for candidate in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "box_train_images" / rel_path,
                      DATA_ROOT / "masks" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
                      DATA_ROOT / "box_train_images" / "box_train_images" / filename, DATA_ROOT / "masks" / "masks" / filename,
                      Path("/tmp") / rel_path):
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    """L2 step 1's rule (the 7B's training rule): no segmentation, a text answer, and every image found."""
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, max_pixels=MAX_PIXELS):
    """The image at the 3B's cap: only an image above it is scaled down (as L0 did at full size)."""
    if img.width * img.height <= max_pixels:
        return img
    h, w = smart_resize(img.height, img.width, 56 * 56, max_pixels)
    return img.resize((w, h), Image.BICUBIC)

assert smart_resize(1024, 1024, 56 * 56, MAX_PIXELS) == (756, 756)
assert held_out({"image_path": "x\\box_train_images\\a_post_7_3.png"}) == held_out({"post_image_path": "a_post_7.png"})
# ── The benchmark's prompts and scorer, and the 3Bs (the same code in steps 3 and 4) ──
# The four prompts, copied from L0 step 2 (which copied them from the DisasterM3 authors' runner)
PROMPTS = dict(
    bearing_body="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option(s) from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    landuse="""Analyze the image to answer the following question. Choose the best option(s) from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    single_choice="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
    relational_reasoning_qa="""Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
)
# (code, subset name in the 499A scorer, task name in the benchmark, task name in train_release.json)
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition", "Disaster Type Recognition"),
         ("BBR", "bearing_body", "Disaster Bearing Bodies Recognition", "Disaster Bearing Bodies Recognition"),
         ("BDC", "building_damage_counting", "Building Damage Counting", "Building Damage Counting"),
         ("DRE", "road_damage_counting", "Road Damage Counting", "Road Damage Counting"),
         ("DSR", "landuse", "Disaster Scene Recognition", "Disaster Scene Recognition"),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning", "relational reasoning")]
COUNTING = ("BDC", "DRE")

def letters(text):
    """The 499A scorer: the set of capital letters left after removing 'THE ANSWER IS' and 'OPTION'."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

def prompt_text(e, subset):
    """(the benchmark prompt, the image keys in order), as L0 step 2's build() makes them."""
    p = e.get("prompts", "")
    p = p[0] if isinstance(p, list) else p
    if subset == "landuse":
        return PROMPTS["landuse"].format(prompt=p, options_str=e["options_str"]), ["pre_image_path"]
    if subset == "relational_reasoning_qa":           # its entries spell the options' field option_str
        return PROMPTS[subset].format(prompt=p, options_str=e["option_str"]), ["image_path"]
    key = "bearing_body" if subset == "bearing_body" else "single_choice"
    return PROMPTS[key].format(prompt=p, options_str=e["options_str"]), ["pre_image_path", "post_image_path"]

def content_of(text, n_images):
    """L0 step 2's layout: single-image prompts put the image first; two-image prompts fill each '<image>' gap."""
    parts = text.split("<image>")
    if len(parts) == 1:
        return [{"type": "image"} for _ in range(n_images)] + [{"type": "text", "text": text}]
    content = []
    for n, part in enumerate(parts):
        if part:
            content.append({"type": "text", "text": part})
        if n < len(parts) - 1:
            content.append({"type": "image"})
    return content

def option_values(e):
    """The five option values of a counting question, as numbers ('5.22%' -> 5.22)."""
    o = e["options_list"]
    o = ast.literal_eval(o) if isinstance(o, str) else o
    return [float(str(x).rstrip("%")) for x in o]

def no_image_rule(values):
    """Never sees the image: largest option above 20 -> the middle value, otherwise the second smallest (benchmark check, 7 Nov)."""
    s = sorted(values)
    return "ABCDE"[values.index(s[2] if s[4] > 20 else s[1])]

def chosen_value(answer, values):
    """The value of the one option an answer names; None if it names no option or several."""
    got = letters(answer)
    if len(got) != 1 or next(iter(got)) not in "ABCDE"[:len(values)]:
        return None
    return values["ABCDE".index(next(iter(got)))]

assert letters("The answer is D.") == {"D"} and no_image_rule([231, 323, 277, 139, 185]) == "A"
assert chosen_value("B", [1, 2, 3, 4, 5]) == 2 and chosen_value("B, C", [1, 2, 3, 4, 5]) is None
assert content_of("x<image>y<image>z", 2)[1] == {"type": "image"} and content_of("q", 1)[0] == {"type": "image"}

def load_3b(which):
    """The base 3B in fp16 on one T4; for 'b' or 'c', its finished adapter merged in."""
    from peft import PeftModel
    from transformers import Qwen2_5_VLForConditionalGeneration
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=torch.float16 if CUDA else torch.float32,
                                                               device_map={"": 0} if CUDA else None, attn_implementation="sdpa")
    if which != "base":
        local = snapshot_download(ADAPTER_REPO, allow_patterns=[f"{which}/final/*"], token=hf_token())
        model = PeftModel.from_pretrained(model, str(Path(local) / which / "final")).merge_and_unload()
    return model.eval()

@torch.inference_mode()
def answer_batch(model, items):
    """items: (prompt text, image paths) -> greedy answers, every sampling setting given, images at the 3B's cap."""
    texts, images = [], []
    for text, paths in items:
        imgs = [sized(Image.open(p).convert("RGB")) for p in paths]
        texts.append(processor.apply_chat_template([{"role": "user", "content": content_of(text, len(imgs))}], tokenize=False,
                                                   add_generation_prompt=True))
        images += imgs
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None,
                         repetition_penalty=1.0)
    return [s.strip() for s in processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]

def answer_safe(model, items):
    """Halves the batch on out-of-memory or an unreadable image; a question that still fails alone gets '' (scored wrong,
    as L0 step 2 scored its unreadable images)."""
    try:
        return answer_batch(model, items)
    except (torch.cuda.OutOfMemoryError, OSError):
        torch.cuda.empty_cache()
        if len(items) == 1:
            return [""]
        half = len(items) // 2
        return answer_safe(model, items[:half]) + answer_safe(model, items[half:])

selection = json.loads(one("L2S3/selection.json", "attach Your Work -> l2-step3-heldout-select").read_text(encoding="utf-8"))
WINNER = selection["winner"]
assert WINNER in ("b", "c"), "step 3's output holds no winner: was it a smoke test?"
BENCH_JSON = one("benchmark_release.json", "attach redwannewaz/disasterm3-bench-mirror-v2")
processor = AutoProcessor.from_pretrained(BASE_MODEL)
processor.tokenizer.padding_side = "left"
print(f"✓ L2 step 4d{'  [SMOKE TEST]' if SMOKE_TEST else ''}: the winner is ({WINNER}): {selection['reason']}")


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✓ L2 step 4d: the winner is (c): primary accuracy 35.95% (b) against 45.12% (c), exact McNemar p = 0.0000


In [3]:
# ── Cell 2b (added 23 Nov): the overflow fix, and the runner-up scored too, one 3B per T4 ──
# The first full run lost 327 answers to a half-precision overflow at the vision part's last block, the layer L2 step 2b
# found in training (19 Nov). As in training, that layer runs in fp32 and the vision part under fp16 mixed precision
# (autocast), which hands its output back to the language model in fp16. These definitions replace cell 2's load_3b,
# answer_batch and answer_safe.
import threading

FP32_LAYER = "visual.blocks.31.mlp.down_proj"    # L2 step 2b's layer
DEVICE = "cuda" if CUDA else "cpu"
ALSO_RUNNER_UP = False       # step 4d: the winner only
OVERFLOWED_BEFORE = ["disaster_type_331", "disaster_type_335", "bearing_body_1641", "bearing_body_1645",
                     "building_damage_counting_1904", "building_damage_counting_1908", "road_damage_counting_1578",
                     "road_damage_counting_1582", "landuse_1451", "landuse_1454", "relational_reasoning_qa_568",
                     "relational_reasoning_qa_569"]   # overflowed in the first full run (2 per task): the smoke test adds them
OVERFLOWED = re.compile(r"!+")                       # what an overflowed answer looks like


def fp32_islands(model, names):
    """L2 step 2's function: these modules run in fp32 with autocast off (their inputs, their fp16 weights and their
    4-bit layers' compute)."""
    mods = dict(model.named_modules())
    for n in names:
        m = mods[n]

        def forward(*args, __orig=m.forward, **kwargs):
            with torch.autocast(DEVICE, enabled=False):
                args = tuple(a.float() if torch.is_tensor(a) and a.is_floating_point() else a for a in args)
                kwargs = {k: v.float() if torch.is_tensor(v) and v.is_floating_point() else v for k, v in kwargs.items()}
                return __orig(*args, **kwargs)

        m.forward = forward
        for sub in m.modules():
            if hasattr(sub, "compute_dtype"):
                sub.compute_dtype = torch.float32
            for p in sub.parameters(recurse=False):
                if p.is_floating_point() and p.dtype != torch.float32:
                    p.data = p.data.float()


def with_fix(model):
    """Step 2b's layer in fp32, inside a vision part run under fp16 autocast, as in training (tested on the PC on a tiny
    random Qwen2.5-VL, 23 Nov: the layer computes in fp32, the image features reach the language model in fp16)."""
    islands = [n for n, _ in model.named_modules() if n.endswith(FP32_LAYER)]
    towers = [m for n, m in model.named_modules() if n.endswith("visual") and hasattr(m, "blocks")]
    assert len(islands) == 1 and len(towers) == 1, f"{len(islands)} of step 2b's layer, {len(towers)} vision parts"
    fp32_islands(model, islands)
    orig = towers[0].forward

    def forward(*args, **kwargs):
        with torch.autocast(DEVICE, dtype=torch.float16):
            return orig(*args, **kwargs)

    towers[0].forward = forward
    return model


def load_3b(which, gpu=0):
    """The base 3B in fp16 on one T4 (GPU `gpu`); for 'b' or 'c', its finished adapter merged in; then the fix."""
    from peft import PeftModel
    from transformers import Qwen2_5_VLForConditionalGeneration
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=torch.float16 if CUDA else torch.float32,
                                                               device_map={"": gpu} if CUDA else None, attn_implementation="sdpa")
    if which != "base":
        local = snapshot_download(ADAPTER_REPO, allow_patterns=[f"{which}/final/*"], token=hf_token())
        model = PeftModel.from_pretrained(model, str(Path(local) / which / "final")).merge_and_unload()
    return with_fix(model).eval()


@torch.inference_mode()
def answer_batch(model, items, proc):
    """items: (prompt text, image paths) -> greedy answers, every sampling setting given, images at the 3B's cap. `proc`
    is this 3B's own processor (one per thread)."""
    texts, images = [], []
    for text, paths in items:
        imgs = [sized(Image.open(p).convert("RGB")) for p in paths]
        texts.append(proc.apply_chat_template([{"role": "user", "content": content_of(text, len(imgs))}], tokenize=False,
                                              add_generation_prompt=True))
        images += imgs
    inputs = proc(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None,
                         repetition_penalty=1.0)
    return [s.strip() for s in proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]


def answer_safe(model, items, proc):
    """Halves the batch on out-of-memory or an unreadable image; a question that still fails alone gets '' (scored wrong,
    as L0 step 2 scored its unreadable images)."""
    try:
        return answer_batch(model, items, proc)
    except (torch.cuda.OutOfMemoryError, OSError):
        torch.cuda.empty_cache()
        if len(items) == 1:
            return [""]
        half = len(items) // 2
        return answer_safe(model, items[:half], proc) + answer_safe(model, items[half:], proc)


RUNNER_UP = "b" if WINNER == "c" else "c"
MODELS = [WINNER] + ([RUNNER_UP] if ALSO_RUNNER_UP else [])
assert not CUDA or torch.cuda.device_count() >= len(MODELS), "Pick GPU T4 x2: each 3B answers on its own GPU"
print(f"✓ The fix: {FP32_LAYER} in fp32, the vision part under fp16 autocast; answering {MODELS}, one per GPU")

✓ The fix: visual.blocks.31.mlp.down_proj in fp32, the vision part under fp16 autocast; answering ['c'], one per GPU


In [4]:
# ── Cell 3: The benchmark's fair building-counting questions, asked with no options ──
NUMBER_PROMPT = """Analyze both the pre-disaster and post-disaster images to answer the following question.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}

Your task is to respond with ONLY the number (e.g., 7). Do not include any explanation or other text."""
FAIR_MAX = 20                  # fair: the largest option is 20 or less (L2 step 4b's split)

with open(BENCH_JSON, encoding="utf-8") as f:
    bench = json.load(f)
bench_index = {}               # every benchmark image by file name (L0 step 2's rule)
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)

def sensor(e):
    return e.get("post_image_type") or e.get("image_type") or "Optical"

code, subset, bench_task, _ = TASKS[2]
assert code == "BDC"
rows = []
for i, e in enumerate(x for x in bench if x.get("task") == bench_task):    # ids as in L2 step 4: the index in the task
    if sensor(e) != "Optical":
        continue
    values = option_values(e)
    if max(values) > FAIR_MAX:
        continue
    prompt = e.get("prompts", "")
    prompt = prompt[0] if isinstance(prompt, list) else prompt
    right = sorted(letters(e["ground_truth_option"]))[0]
    rows.append({"id": f"{subset}_{i}", "prompt": prompt, "text": NUMBER_PROMPT.format(prompt=prompt),
                 "images": [bench_index.get(name_of(e[k])) for k in ("pre_image_path", "post_image_path")],
                 "values": values, "truth_letter": right, "truth_value": values["ABCDE".index(right)]})
Q = pd.DataFrame(rows)
Q["image_missing"] = Q["images"].map(lambda ps: any(p is None for p in ps))
assert len(Q) == 2002, f"{len(Q)} fair questions, expected L2 step 4b's 2,002"
assert "Options" not in Q["text"].iloc[0] and "{" not in Q["text"].iloc[0]
if SMOKE_TEST:
    Q = Q.head(16).reset_index(drop=True)
print(f"✓ {len(Q):,} fair building-counting questions{' [SMOKE TEST]' if SMOKE_TEST else ''}; "
      f"{int(Q['image_missing'].sum())} with an image missing (scored wrong)")
print(Q["text"].iloc[0])


✓ 2,002 fair building-counting questions; 0 with an image missing (scored wrong)
Analyze both the pre-disaster and post-disaster images to answer the following question.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: Enumerate the total intact buildings that survived the disaster event.

Your task is to respond with ONLY the number (e.g., 7). Do not include any explanation or other text.


In [5]:
# ── Cell 4: The 3B (c) answers, written as it goes (resumable) ──
from tqdm.auto import tqdm

done = {}
if ANSWERS_PATH.exists():
    with open(ANSWERS_PATH, encoding="utf-8") as f:
        for r in map(json.loads, f):
            done[r["id"]] = r["answer"]
model = load_3b(WINNER, gpu=0)
proc = AutoProcessor.from_pretrained(BASE_MODEL)
proc.tokenizer.padding_side = "left"
todo = Q[~Q["id"].isin(list(done)) & ~Q["image_missing"]]
t0 = time.time()
with open(ANSWERS_PATH, "a", encoding="utf-8") as fh:
    for start in tqdm(range(0, len(todo), BATCH), desc=f"3B ({WINNER})"):
        batch = todo.iloc[start:start + BATCH]
        for qid, answer in zip(batch["id"], answer_safe(model, list(zip(batch["text"], batch["images"])), proc)):
            fh.write(json.dumps({"id": qid, "model": WINNER, "answer": answer, "fix": FP32_LAYER}) + "\n")
            done[qid] = answer
        fh.flush()
RUN_MIN = (time.time() - t0) / 60
print(f"✓ {len(done):,} answered in {RUN_MIN:.1f} min; the first: {[done[q] for q in list(done)[:8]]}")


model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

3B (c):   0%|          | 0/251 [00:00<?, ?it/s]

✓ 2,002 answered in 59.9 min; the first: ['1', '1', '2', '1', '5', '11', '1', '2']


In [6]:
# ── Cell 5: The numbers against the true counts, and the files ──
import numpy as np
import peft, transformers
NUM = re.compile(r"\d+(?:\.\d+)?")

def number(answer):
    m = NUM.search(str(answer))
    return float(m.group()) if m else np.nan

def nearest_letter(values, n):
    """The option nearest the number (the first one on a tie), as C4's counts were snapped in E7."""
    return "ABCDE"[min(range(len(values)), key=lambda k: abs(values[k] - n))] if n == n else ""

Q["answer"] = Q["id"].map(done).fillna("")
Q["number"] = Q["answer"].map(number)
Q["abs_error"] = (Q["number"] - Q["truth_value"]).abs()
Q["exact"] = Q["abs_error"] == 0
Q["nearest_letter"] = [nearest_letter(v, n) for v, n in zip(Q["values"], Q["number"])]
Q["nearest_right"] = Q["nearest_letter"] == Q["truth_letter"]
Q["overflowed"] = Q["answer"].map(lambda a: bool(OVERFLOWED.fullmatch(a)))
asked = Q[~Q["image_missing"]]
pct = lambda s: round(100 * float(s.mean()), 2)
summary = {
    "questions": len(Q), "asked": len(asked), "smoke_test": SMOKE_TEST, "model": WINNER, "fix": FP32_LAYER,
    "with_a_number_pct": pct(asked["number"].notna()), "letter_only_answers": int(asked["answer"].str.fullmatch(r"[A-E]").sum()),
    "exact_pct": pct(Q["exact"]), "within_1_pct": pct(Q["abs_error"] <= 1),
    "nearest_option_right_pct": pct(Q["nearest_right"]),
    "mean_abs_error": round(float(asked["abs_error"].mean()), 2), "median_abs_error": float(asked["abs_error"].median()),
    "overflowed": int(Q["overflowed"].sum()), "minutes": round(RUN_MIN, 1),
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__},
}
Q.drop(columns=["text", "images"]).to_csv(OUT_DIR / "per_question.csv", index=False)
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")
print(json.dumps(summary, indent=1))
print(Q[["id", "truth_value", "answer", "number", "nearest_right"]].head(16).to_string(index=False))


{
 "questions": 2002,
 "asked": 2002,
 "smoke_test": false,
 "model": "c",
 "fix": "visual.blocks.31.mlp.down_proj",
 "with_a_number_pct": 99.9,
 "letter_only_answers": 0,
 "exact_pct": 21.88,
 "within_1_pct": 40.16,
 "nearest_option_right_pct": 35.56,
 "mean_abs_error": 4.12,
 "median_abs_error": 2.0,
 "overflowed": 0,
 "minutes": 59.9,
 "versions": {
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "peft": "0.22.0"
 }
}
                          id  truth_value                                                                 answer  number  nearest_right
  building_damage_counting_8          2.0                                                                      1     1.0           True
  building_damage_counting_9          2.0                                                                      1     1.0           True
 building_damage_counting_13          3.0                                                                      2     2.0           True
 building_damage_count